# KYC / bank-statement PDF trial: step by step

Run top to bottom. Everything is imported from `utils/` so you can edit a function in
`utils/common.py`, re-run the **reload** cell and try again.

**Never save client data in this notebook.** Before `git commit`: *Clear All Outputs*
(VS Code: `...` > Clear All Outputs). Client PDFs go in `data/` (git-ignored).

## 1. Imports + pick the OCR engine
`rapid` in the rapid venv, `paddle` in the paddle venv.

In [ ]:
import importlib
import pandas as pd
from PIL import Image
from utils import common

ENGINE = "rapid"   # "rapid" or "paddle"

if ENGINE == "rapid":
    from utils import ocr_rapid as ocr_mod
else:
    from utils import ocr_paddle as ocr_mod
ocr_page = ocr_mod.ocr_page
pd.set_option("display.max_rows", 200, "display.max_colwidth", 80)

Reload after editing `utils/*.py` (settings you overrode below must be set again):

In [ ]:
importlib.reload(common); importlib.reload(ocr_mod); ocr_page = ocr_mod.ocr_page

## 2. Settings you may want to change
All settings are constants at the top of `utils/common.py`. Override here for a quick try:

In [ ]:
common.DPI = 250                  # try 300 for poor scans
common.DESKEW = True
common.BINARISE = False           # benchmark: binarising hurts RapidOCR
common.STATEMENT_MIN_SCORE = 0.45
common.CACHE_DIR = "outputs/_ocr_cache"   # None = always re-OCR
# common.COLUMN_SYNONYMS["debit"].append("withdrawal (dr)")
# common.FIELDS["app_id"]["labels"].append("proposal no")

## 3. Pick a PDF and render one page

In [ ]:
PDF = "tests/fixtures/synthetic_scanned.pdf"   # e.g. "data/file01.pdf"
PAGE = 3                                        # 1-based

print("pages:", common.page_count(PDF))
img = common.render_page(PDF, PAGE - 1)
print(img.shape)
Image.fromarray(img).resize((img.shape[1] // 3, img.shape[0] // 3))

## 4. Text layer or OCR?
Digital pages have a text layer (exact, fast). Scanned pages are deskewed and OCRed.

In [ ]:
tl = common.text_layer_words(PDF, PAGE - 1)
print("text-layer chars:", sum(len(w["text"]) for w in tl), "(>=", common.MIN_TEXT_CHARS, "means use text layer)")

clean, angle = common.preprocess(img)
print("skew angle:", round(angle, 2))
Image.fromarray(clean).resize((clean.shape[1] // 3, clean.shape[0] // 3))

In [ ]:
%%time
pg = common.get_page_words(PDF, PAGE - 1, ocr_page, ENGINE)   # text layer, cache, or OCR
words = pg["words"]
print(pg["source"], "words:", len(words), "from cache:", pg["cached"])
pd.DataFrame(words).head(40)

Raw OCR without cache/preprocessing (to compare):

In [ ]:
# words_raw = ocr_page(img)
# pd.DataFrame(words_raw).head(40)

## 5. Lines + classification

In [ ]:
res, lines = common.classify_page(words)
print(res["label"], "st =", res["statement_score"], "form =", res["form_score"])
print(res["features"])
for ln in lines[:30]:
    print(common.line_text(ln))

## 6. Parse the statement table on this page
Header row (credit/debit/balance) sets the columns; each row gets the balance check.

In [ ]:
hi, hline = common.header_line(lines)
print("header:", common.line_text(hline) if hline else None)
rows, cols, opening = common.parse_page_table(lines)
print("columns:", [(c["type"], round(c["x0"]), round(c["x1"])) for c in cols or []], "opening:", opening)
rate, checked = common.balance_check(rows, opening)
print("rows:", len(rows), "balance ok rate:", rate, "checked:", checked)
pd.DataFrame(rows)

Rows that failed the balance check (usually OCR digit errors; look at `raw`):

In [ ]:
pd.DataFrame([r for r in rows if r["balance_ok"] == "fail"])

## 7. Field extraction on the form page

In [ ]:
FORM_PAGE = 1
fpg = common.get_page_words(PDF, FORM_PAGE - 1, ocr_page, ENGINE)
_, flines = common.classify_page(fpg["words"])
common.extract_fields(flines, page_no=FORM_PAGE)

## 8. One whole PDF
Pages are classified, consecutive statement pages are grouped, months are counted.

In [ ]:
r = common.process_pdf(PDF, ocr_page, ENGINE)
print(r["summary"])
pd.DataFrame(r["pages"]).drop(columns=["text", "features"])

In [ ]:
pd.DataFrame(r["statements"])

In [ ]:
pd.DataFrame(r["transactions"]).head(20)

## 9. Whole folder
Writes `outputs/<engine>/<pdf>.json`, `<pdf>_transactions.csv`, `summary.csv` and `outputs/manual_review.csv`.

In [ ]:
DATA = "tests/fixtures"   # "data" for client PDFs
summary = common.run_folder(ocr_page, ENGINE, DATA, "outputs")
pd.DataFrame(summary)

## 10. Accuracy vs your manual checks
Fill the `*_true` columns of `outputs/manual_review.csv` by hand, then:

In [ ]:
import os
mr = "outputs/manual_review.csv"
if os.path.exists(mr):
    common.evaluate(f"outputs/{ENGINE}/summary.csv", mr)